# 5. 근거 기반 API 통합 테스트

**시나리오:** 지원 질문과 미지원 질문을 실제 FastAPI 경계에서 비교합니다.

**학습 목표:** `TestClient`, dependency injection, citation과 `insufficient_evidence` 안전 계약을 함께 검증합니다.

## 중요 변수·함수

- `create_app(services)`: fixture를 주입해 HTTP와 workflow를 함께 시험합니다.
- `/query`: Pydantic 검증 후 canonical `run_policy_query()`를 호출합니다.
- `citations`: 답변에 실제 사용된 근거 식별자입니다.

In [ ]:
# 이 학습 Notebook은 외부 API/DB를 사용하지 않는 fixture 모드로 고정합니다.
import os
os.environ["APP_MODE"] = "fixture"

# Notebook 위치에서 실행해도 repository의 canonical app을 가져옵니다.
from pathlib import Path
import sys

_repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

# 실제 서버 없이 FastAPI 요청/응답 경계를 실행합니다.
from fastapi.testclient import TestClient
from week1.app import create_app, create_fixture_services

client = TestClient(create_app(create_fixture_services()))
supported = client.post('/query', json={'question': 'vacation request notice'}).json()
unsupported = client.post('/query', json={'question': 'office wifi password'}).json()

In [ ]:
# 두 경로의 안전 불변조건을 비교합니다.
assert supported['status'] == 'answered' and supported['citations']
assert unsupported['status'] == 'insufficient_evidence'
assert unsupported['answer'] is None and unsupported['citations'] == []
{'supported': supported['status'], 'unsupported': unsupported['status']}

## 예측 과제와 해석

**예측 과제:** HTTP 200만 검사하는 테스트가 놓치는 오류를 두 가지 적으세요.

**해석:** 성공 코드는 업무 정답을 보장하지 않습니다. 상태·답변·인용·trace를 함께 검사해야 grounded RAG 계약을 검증할 수 있습니다.